# Mistral Air Quality Chat

Reference notebook for the application's air-quality LLM workflow.

It follows the same logic used by the Streamlit app:

- reads the API key from `.confing` or `MISTRAL_API_KEY`;
- loads the current scraped measurements and the latest predictions;
- formats both tables for prompt context;
- builds a user-question prompt for chat;
- builds the initial general-summary prompt.

## Workflow

1. Locate the project root and import the app's LLM helpers.
2. Load the current and predicted air-quality tables.
3. Inspect the markdown context passed to the model.
4. Preview both prompt types used by the app.
5. Keep real Mistral calls commented out to avoid accidental API usage.

## Setup

Import the local path helper and the data library used for table previews.

In [ ]:
# If this is running in a fresh environment:
# %pip install mistralai pandas

from pathlib import Path
import sys

import pandas as pd

## App Imports

Find the project root from the current notebook location, then import the same helper functions used by the application.

In [ ]:
def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "app" / "air_quality_llm.py").exists():
            return candidate
    raise FileNotFoundError("Could not find app/air_quality_llm.py from the current directory.")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "app"))

from air_quality_llm import (  # noqa: E402
    CURRENT_PATH,
    PREDICTIONS_PATH,
    ask_air_quality,
    build_dynamic_context,
    build_general_summary_prompt,
    build_user_prompt,
    format_markdown_table,
    load_air_table,
    summarize_air_quality,
)

ROOT

## Load Air-Quality Tables

Load the current scraped data and the latest predictions, then show a small preview of each table.

In [ ]:
current_df = load_air_table(CURRENT_PATH)
predictions_df = load_air_table(PREDICTIONS_PATH)

display(current_df.head())
display(predictions_df.head())

## Prompt Tables

Each pollutant cell is formatted as `value ug/m3 (quality)`. Missing values are shown as `ND`. The table titles stay in Spanish because they are part of the production prompt context.

In [ ]:
print(format_markdown_table(current_df, "TABLA ACTUAL SCRAPEADA"))

In [ ]:
print(format_markdown_table(predictions_df, "TABLA DE PREDICCIONES"))

## Full Dynamic Context

This is the dynamic block added to each model call: current measurements plus predictions.

In [ ]:
print(build_dynamic_context())

## Chat Prompt

Build the prompt for a user question. The example question is kept in Spanish because the app assistant is instructed to answer users in Spanish.

In [ ]:
example_question = "Que zona esta peor ahora para NO2 y que predice el modelo?"
print(build_user_prompt(example_question)[:3500])

## Initial Summary Prompt

Build the prompt used to generate the general air-quality summary shown when the app starts.

In [ ]:
print(build_general_summary_prompt()[:3500])

## Real Mistral Calls

These calls are intentionally commented out to avoid spending API calls by accident.

In [ ]:
# Chat:
# answer = ask_air_quality("Que contaminante preocupa mas ahora mismo en Valencia?")
# print(answer)

# Initial general summary:
# summary = summarize_air_quality()
# print(summary)